In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql import functions as F
import os


In [ ]:
try:
    spark.stop()
except:
    pass


In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_Encounter")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.config("spark.sql.session.timeZone", "UTC")
.master("local[*]")
.getOrCreate())


In [ ]:
silver_base_path = "../../data_lake/silver/silver_encounter_bundle/"
gold_base_path = "../../data_lake/gold/dim_encounter/"
gold_dimpatient = "../../data_lake/gold/dim_patient/"
gold_dimpractitioner = "../../data_lake/gold/dim_practitioner/"
gold_dimorganization = "../../data_lake/gold/dim_organization/"
gold_dimlocation = "../../data_lake/gold/dim_location/"
gold_dimdate = "../../data_lake/gold/dim_date/"


In [ ]:
df_encounter = spark.read.format("parquet").load(silver_base_path)
df_dimpatient = spark.read.format("parquet").load(gold_dimpatient)
df_dimpractitioner = spark.read.format("parquet").load(gold_dimpractitioner)
df_dimorganization = spark.read.format("parquet").load(gold_dimorganization)
df_dimlocation = spark.read.format("parquet").load(gold_dimlocation)
df_dimdate = spark.read.format("parquet").load(gold_dimdate)


In [ ]:
df_inter = (df_encounter.alias("enc")
    .join(df_dimpatient.alias("pat"), F.regexp_replace(col("enc.patient_id"), "^urn:uuid:", "") == col("pat.patient_id"), "left")
    .join(df_dimpractitioner.alias("prac"), col("enc.participant_individual_npi") == col("prac.npi"), "left")
    .join(df_dimorganization.alias("org"), col("enc.organization_org_id") == col("org.organization_id"), "left")
    .join(df_dimlocation.alias("loc"), col("enc.location_id") == col("loc.location_id"), "left")
    .join(df_dimdate.alias("d_start"), col("enc.period_start").cast("date") == col("d_start.date"), "left")
    .join(df_dimdate.alias("d_end"), col("enc.period_end").cast("date") == col("d_end.date"), "left")
    .select(
        F.conv(F.substring(F.md5(col("enc.encounter_id")), 1, 15), 16, 10).cast("bigint").alias("encounter_key"),
        col("enc.encounter_id"),
        col("pat.patient_key"),
        col("prac.practitioner_key"),
        col("org.organization_key"),
        col("loc.location_key"),
        col("enc.status"),
        col("enc.class_code"),
        col("enc.type_code"),
        col("enc.type_display"),
        col("d_start.date_key").alias("period_start_date_key"),
        col("d_end.date_key").alias("period_end_date_key"),
        col("enc.period_start"),
        col("enc.period_end"),
        col("enc.reason_code"),
        col("enc.reason_display"),
        col("enc.hospitalization_discharge_disposition_code"),
        col("enc.hospitalization_discharge_disposition_display"),
        col("enc.participant_type_code"),
        col("enc.participant_type_display"),
        F.current_timestamp().alias("gold_timestamp")
    )
)


In [ ]:
df_inter.write.mode("overwrite").format("parquet").save(gold_base_path)


In [ ]:
spark.stop()
